<div style="text-align: center;">
  <h1 style="text-align: center;">Reto 2 — SOLUCIÓN de referencia</h1>
  <h3 style="text-align: center;">Master IA &amp; Data Science · EBIS Business School · Bloque B</h3>
  <p style="text-align: center;">Workshop: Prototipado de Agentes</p>
</div>

> ⚠️ **Este notebook contiene la solución completa**. Es la referencia del instructor — los alumnos trabajan sobre `2_Reto_asistente.ipynb` (con TODOs).

## Resumen de la solución

- Ingesta de los 3 PDFs de guías de viaje (Japón, Tailandia, Portugal)
- Tools: `search_kb` (Chroma con embeddings OpenAI) + `fetch_section` (lectura directa de PDF)
- Salida tipada: `Respuesta(texto, citas, confianza)`
- Output guardrail: exige al menos una cita real
- Input guardrail: rechaza preguntas off-topic
- Eval set automatizado (5 casos, citation rate)

## Setup

In [ ]:
!uv pip install --system -q --upgrade openai-agents==0.4.1 nest_asyncio==1.6.0 "chromadb>=0.5.20" pypdf==4.3.1 pyyaml==6.0.2 "typing_extensions>=4.13.0"

# ⚠️ Tras instalar/actualizar paquetes, REINICIA el kernel (Kernel → Restart Kernel).

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
from pathlib import Path

MANUALS_DIR = Path("data/manuals")
CHROMA_DIR = Path("data/chroma")
EVAL_FILE = Path("tests/cases.yaml")

MANUALS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE.parent.mkdir(parents=True, exist_ok=True)

print("✅ Setup completado.")
print("Guías en:", MANUALS_DIR.resolve())
print("Ficheros:", [p.name for p in MANUALS_DIR.glob("*.pdf")])

## TODO 1 — Ingesta de guías en Chroma ✅

Una entrada por página. Metadatos `{manual, pagina}`. ID determinista para que `upsert` sea idempotente.

In [ ]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

import chromadb
from chromadb.api.types import EmbeddingFunction
from openai import OpenAI
from pypdf import PdfReader


class OpenAIEmbed(EmbeddingFunction):
    """Embedding function usando openai>=1.0 (el de chromadb 0.5.x usa la API legacy)."""
    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI()
        self.model = model

    def __call__(self, input: list[str]) -> list[list[float]]:
        res = self.client.embeddings.create(model=self.model, input=input)
        return [d.embedding for d in res.data]


client_chroma = chromadb.Client(
    settings=chromadb.Settings(anonymized_telemetry=False),
)
coleccion = client_chroma.get_or_create_collection(
    "guias_viaje",
    embedding_function=OpenAIEmbed(),
)


def ingestar_guias(directorio: Path, force: bool = False) -> int:
    """Ingesta todas las páginas de los PDFs en `directorio` a Chroma.
    Si la colección ya tiene documentos y `force=False`, no re-ingesta.
    """
    if coleccion.count() > 0 and not force:
        print(f"⏩  Colección ya tiene {coleccion.count()} docs. Skip (pasa force=True para re-ingestar).")
        return coleccion.count()
    docs, metas, ids = [], [], []
    for pdf in sorted(directorio.glob("*.pdf")):
        reader = PdfReader(str(pdf))
        for i, page in enumerate(reader.pages, start=1):
            texto = (page.extract_text() or "").strip()
            if not texto:
                continue
            docs.append(texto)
            metas.append({"manual": pdf.name, "pagina": i})
            ids.append(f"{pdf.name}::{i}")
    if docs:
        coleccion.upsert(documents=docs, metadatas=metas, ids=ids)
    return len(docs)


total = ingestar_guias(MANUALS_DIR)
print(f"✅ Total páginas en colección: {total}")

## TODO 2 — Tool `search_kb` ✅

In [ ]:
from agents import function_tool
from pydantic import BaseModel

class Fragmento(BaseModel):
    texto: str
    manual: str
    pagina: int

@function_tool
def search_kb(query: str, k: int = 4) -> list[Fragmento]:
    """Busca los k chunks más relevantes en las guías de viaje.

    Args:
        query: Pregunta del viajero en lenguaje natural.
        k: Número de fragmentos a devolver (por defecto 4).
    """
    print(f"  🔎 search_kb({query[:60]!r}, k={k})")
    res = coleccion.query(query_texts=[query], n_results=k)
    fragmentos = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        fragmentos.append(Fragmento(
            texto=doc,
            manual=meta["manual"],
            pagina=int(meta["pagina"]),
        ))
    return fragmentos

## TODO 3 — Tool `fetch_section` ✅

In [ ]:
@function_tool
def fetch_section(manual: str, pagina: int) -> str:
    """Lee la página `pagina` (1-indexed) del PDF `data/manuals/{manual}`.

    Args:
        manual: Nombre del fichero PDF (ej: 'guia_japon.pdf').
        pagina: Número de página (empezando en 1).
    """
    print(f"  📄 fetch_section({manual}, p{pagina})")
    path = MANUALS_DIR / manual
    if not path.exists():
        return f"[error] Guía no encontrada: {manual}"
    reader = PdfReader(str(path))
    if pagina < 1 or pagina > len(reader.pages):
        return f"[error] Página {pagina} fuera de rango (1-{len(reader.pages)})"
    return reader.pages[pagina - 1].extract_text() or "[página vacía]"

## TODO 4 — Salida tipada + Guardrails ✅

In [ ]:
from agents import (
    Agent, output_guardrail, input_guardrail,
    GuardrailFunctionOutput, RunContextWrapper,
)
from pydantic import Field

class Cita(BaseModel):
    manual: str
    pagina: int

class Respuesta(BaseModel):
    texto: str = Field(description="Respuesta en lenguaje natural, 3-5 frases")
    citas: list[Cita] = Field(description="Al menos una cita a una guía de viaje")
    confianza: float = Field(description="Confianza 0-1 basada en la calidad de los fragmentos")

@output_guardrail
async def exigir_citas(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: Respuesta,
) -> GuardrailFunctionOutput:
    """La respuesta tiene que llevar al menos una cita real."""
    citas_validas = [c for c in output.citas if c.manual and c.pagina > 0]
    if not citas_validas:
        raise ValueError("⛔ Respuesta sin citas válidas (guía + página).")
    return GuardrailFunctionOutput(
        output_info=f"{len(citas_validas)} citas válidas",
        tripwire_triggered=False,
    )

@input_guardrail
async def rechazar_offtopic(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text: str,
) -> GuardrailFunctionOutput:
    """Rechaza preguntas claramente fuera del dominio de viajes."""
    keywords_on_topic = [
        "viaje", "visado", "visa", "vuelo", "hotel", "moneda", "clima", "temperatura",
        "japón", "japon", "tailandia", "portugal", "lisboa", "tokio", "bangkok",
        "presupuesto", "coste", "euro", "yen", "baht", "transporte", "tren", "metro",
        "destino", "días", "semana", "verano", "invierno", "primavera", "otoño",
        "gastronomía", "comida", "restaurante", "seguridad", "vacuna",
    ]
    if not any(k in input_text.lower() for k in keywords_on_topic):
        raise ValueError("⛔ Pregunta fuera del dominio de viajes.")
    return GuardrailFunctionOutput(
        output_info="Pregunta on-topic",
        tripwire_triggered=False,
    )

## TODO 5 — Construir el agente ✅

In [ ]:
SYSTEM_PROMPT = """
Eres TravelMind, el asistente de viajes de EBIS Business School.
Tu única fuente de verdad son las guías de viaje internas accesibles vía las tools.
Reglas:
1. Antes de responder, llama a `search_kb` con la pregunta del viajero.
2. Si necesitas más contexto, usa `fetch_section` para leer la página completa.
3. Responde en 3-5 frases. Sé concreto y accionable.
4. Incluye SIEMPRE al menos una cita (guía + página). Si no hay fuente, dilo.
5. Si la pregunta no está cubierta por las guías, indícalo claramente.
""".strip()

agente_guias = Agent(
    name="TravelMind Guías",
    instructions=SYSTEM_PROMPT,
    tools=[search_kb, fetch_section],
    output_type=Respuesta,
    output_guardrails=[exigir_citas],
    input_guardrails=[rechazar_offtopic],
)
print("✅ Agente listo")

## TODO 6 — Probar el agente ✅

In [ ]:
from agents import Runner, trace

PREGUNTA = "¿Cuántos días puede quedarse un español en Tailandia sin visado?"

with trace("reto2 prueba manual") as t:
    resultado = Runner.run_sync(agente_guias, PREGUNTA)
    trace_id = t.trace_id

r = resultado.final_output
print("📝", r.texto)
print()
print("📑 Citas:")
for c in r.citas:
    print(f"   - {c.manual} · p{c.pagina}")
print(f"\n🔎 Confianza: {r.confianza}")
print(f"\n🔗 Traza: https://platform.openai.com/traces/trace?trace_id={trace_id}")

## TODO 7 — Eval set ✅

In [ ]:
import time
import yaml

def correr_eval():
    casos = yaml.safe_load(EVAL_FILE.read_text())
    pasados = 0
    citas_correctas = 0
    latencias = []
    for caso in casos:
        t0 = time.perf_counter()
        try:
            res = Runner.run_sync(agente_guias, caso["pregunta"])
            r = res.final_output
        except Exception as e:
            print(f"❌ {caso['id']}: error {type(e).__name__}: {e}")
            continue
        lat = time.perf_counter() - t0
        latencias.append(lat)

        keyword_ok = caso["keyword"].lower() in r.texto.lower()
        cita_ok = bool(r.citas)
        manual_ok = any(
            c.manual == caso.get("manual_esperado") for c in r.citas
        ) if caso.get("manual_esperado") else cita_ok

        if keyword_ok and cita_ok:
            pasados += 1
        if cita_ok and manual_ok:
            citas_correctas += 1

        status = "✅" if (keyword_ok and cita_ok) else "❌"
        print(f"{status} {caso['id']} ({lat:.1f}s)")
        print(f"    keyword '{caso['keyword']}' {'sí' if keyword_ok else 'NO'} · "
              f"citas {len(r.citas)} · manual {'OK' if manual_ok else 'mal'}")
        if r.citas:
            print(f"    → {r.citas[0].manual} p{r.citas[0].pagina}")
    n = len(casos)
    print(f"\n📊 Pasados: {pasados}/{n}")
    print(f"📑 Citation rate: {citas_correctas}/{n}")
    if latencias:
        print(f"⏱  Latencia media: {sum(latencias)/len(latencias):.1f}s")

correr_eval()

## Notas del instructor

- Coste estimado de un eval set completo: ~1 centavo (5 preguntas con `gpt-4o-mini` + embeddings `text-embedding-3-small`).
- La primera ejecución de la ingesta genera ~27 embeddings (9 páginas × 3 guías) — luego Chroma reutiliza.
- Si `correr_eval()` falla por timeouts, es probable que el modelo no esté llamando a `search_kb` — revisa el system prompt.
- Para limpiar Chroma y re-ingestar: `coleccion.delete(where={})` o llama `ingestar_guias(MANUALS_DIR, force=True)`.
- Las trazas de todas las ejecuciones quedan en `platform.openai.com/traces`.
- Los PDFs sintéticos se pueden regenerar con `python gen_pdfs.py` desde el directorio `notebooks/`.